In [1]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

In [2]:
dataset = load_dataset("OpenAssistant/oasst1")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

In [3]:
df = dataset["train"].to_pandas()

In [4]:
print(df.columns)
print(df.head())
print(df.shape)

Index(['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role',
       'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic',
       'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis',
       'labels'],
      dtype='object')
                             message_id                             parent_id  \
0  6ab24d72-0181-4594-a9cd-deaf170242fb                                  None   
1  c8e83833-ecbc-44fe-b6db-735228c25a1c  6ab24d72-0181-4594-a9cd-deaf170242fb   
2  6708c47f-05c9-4346-b3d2-40b2bd24fde4  c8e83833-ecbc-44fe-b6db-735228c25a1c   
3  343ee2d4-87ae-41fd-a768-bdd65959dc4a  6ab24d72-0181-4594-a9cd-deaf170242fb   
4  18145bf4-37fd-4ac0-80f5-6108b5f2b365  343ee2d4-87ae-41fd-a768-bdd65959dc4a   

                                user_id                      created_date  \
0  c3fe8c76-fc30-4fa7-b7f8-c492f5967d18  2023-02-05T14:23:50.983374+00:00   
1  2c96e467-66f0-4be7-9693-bda51356a424  2023-02-06T13:50:44.657083+00:00   
2  2c96e4

In [5]:
df = df[df["lang"] == "en"].copy()

print(df.shape)

(39283, 18)


In [6]:
df = df[
    (df["deleted"] == False) &
    (df["review_result"] == True)
].copy()

print(df.shape)

(37783, 18)


In [8]:
message_lookup = df.set_index("message_id").to_dict("index")

In [9]:
pairs = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    # We only want assistant replies
    if row["role"] != "assistant":
        continue

    parent_id = row["parent_id"]

    # Skip root messages
    if pd.isna(parent_id):
        continue

    # Parent must exist
    if parent_id not in message_lookup:
        continue

    parent = message_lookup[parent_id]

    # Parent must be from the user
    if parent["role"] != "prompter":
        continue

    pairs.append({
        "prompt": parent["text"].strip(),
        "response": row["text"].strip()
    })

  0%|          | 0/37783 [00:00<?, ?it/s]

In [11]:
chat_df = pd.DataFrame(pairs)

print(chat_df.shape)

chat_df.head()

(23062, 2)


,prompt,response
0,Can you write a short introduction about the r...,"""Monopsony"" refers to a market structure where..."
1,Can you write a short introduction about the r...,Monopsony is a market structure in which there...
2,Can you write a short introduction about the r...,Monopsony refers to a market structure where t...
3,What can be done at a regulatory level to ensu...,Here are some potential regulatory options to ...
4,What can be done at a regulatory level to ensu...,Regulatory intervention can be used to address...


In [14]:
chat_df.iloc[400]

prompt      how would i merge different sheets in an excel...
response    You can do this with the consolidate feature i...
Name: 400, dtype: object

In [15]:
chat_df = chat_df.drop_duplicates()

chat_df = chat_df[
    (chat_df["prompt"].str.len() > 5) &
    (chat_df["response"].str.len() > 5)
]

chat_df = chat_df.reset_index(drop=True)

print(chat_df.shape)

(22813, 2)


In [16]:
chat_df.sample(100)

,prompt,response
5837,Is this an approximation or the exact value?,"No, the exact value is 3360,2 + 27/ 990."
21640,Could you provide a mathematical proof for why...,Assume for the sake of contradiction that ther...
14999,How about a haiku?,"In heaven azure, \nGreat blades of flame clash..."
5796,Recite the lyrics to the Pokemon theme please.,Here are the lyrics of the Pokemon theme.\n\n[...
4009,How would I harden the security of a Linux sys...,To harden the security of a Linux system using...
...,...,...
64,"Great, which song are these lyrics from, and w...","These lyrics come from the ragtime song ""Junk ..."
16439,"Could you rewrite that program, but minimizing...",Yes Sir\n```js\nfunction Viterbi(data) {\n ...
395,how would i merge different sheets in an excel...,To merge multiple sheets in an Excel file into...
13932,Thank you for writing the function. Can you ex...,First the function retrieves the payload from ...


In [17]:
chat_df.shape

(22813, 2)

In [19]:
for i in chat_df.sample(4).index:
    print("=" * 100)
    print("Prompt:")
    print(chat_df.loc[i, "prompt"])
    print()
    print("Response:")
    print(chat_df.loc[i, "response"])

Prompt:
Wouldn't that have been the first time his quest interfered with his life? Why did you say it interfered 'again'?

Response:
The first time that his quest interfered with his life, Bob chose to abandon quest to live life as a common office worker, which led to the death of his former sect master at the hand of Lord Zhao.

It is a decision which weighs heavily on Bob's mind, and it is this guilt which drives him to practice his martial art skills night after night in order to avenge his master.
Prompt:
I'm doing a report on Friedrich Hayek. Can you tell me about him?  Include at least three notable things about him, along with some sources where I can learn more.

Response:
Friedrich August von Hayek (May 8, 1899 - March 23, 1992), also known as F. A. Hayek.

1. Friedrich Hayek was an Austrian-British economist and philosopher who is best known for his work on classical liberalism and free market economics.

2. Hayek was awarded the Nobel Memorial Prize in Economic Sciences in 1

In [20]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k")

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-f7dfac4afe5(…):   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

data/test_gen-00000-of-00001-3d4cd830914(…):   0%|          | 0.00/80.4M [00:00<?, ?B/s]

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

In [22]:
print(dataset)

# print(dataset["train"][0])

DatasetDict({
    train_sft: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 207865
    })
    test_sft: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 23110
    })
    train_gen: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 256032
    })
    test_gen: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 28304
    })
})


In [23]:
train = dataset["train_sft"]

In [25]:
print(train[0]["messages"])

[{'content': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?", 'role': 'user'}, {'content': 'This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.', 'role': 'assistant'}, {'content': 'Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?', 'role': 'user'}, {'content': "Sure, here are the step

In [26]:
pairs = []

for sample in tqdm(train):

    messages = sample["messages"]

    for i in range(len(messages) - 1):

        if (
            messages[i]["role"] == "user"
            and messages[i + 1]["role"] == "assistant"
        ):

            pairs.append({
                "prompt": messages[i]["content"].strip(),
                "response": messages[i + 1]["content"].strip()
            })

  0%|          | 0/207865 [00:00<?, ?it/s]

In [27]:
ultrachat_df = pd.DataFrame(pairs)

print(ultrachat_df.shape)

ultrachat_df.head()

(657794, 2)


,prompt,response
0,These instructions apply to section-based them...,This feature only applies to Collection pages ...
1,Can you guide me through the process of enabli...,"Sure, here are the steps to enable the seconda..."
2,Can you provide me with a link to the document...,I don't have access to your store's theme info...
3,Can you confirm if this feature also works for...,The secondary image hover feature may or may n...
4,Which famous landmarks should I visit in Londo...,1. Leadenhall Market - a beautiful indoor mark...


In [33]:
ultrachat_df = ultrachat_df.drop_duplicates()

ultrachat_df = ultrachat_df.dropna()

ultrachat_df = ultrachat_df[
    (ultrachat_df["prompt"].str.len() > 5)
    &
    (ultrachat_df["response"].str.len() > 5)
]

ultrachat_df = ultrachat_df.reset_index(drop=True)

print(ultrachat_df.shape)

(657743, 2)


In [34]:
ultrachat_df = ultrachat_df.dropna()
ultrachat_df = ultrachat_df.drop_duplicates()

ultrachat_df = ultrachat_df[
    (ultrachat_df["prompt"].str.len() > 10) &
    (ultrachat_df["response"].str.len() > 10)
]

ultrachat_df = ultrachat_df.reset_index(drop=True)

In [35]:
ultrachat_df = ultrachat_df[
    ultrachat_df["response"].str.split().str.len() >= 5
]

In [36]:
ultrachat_df = ultrachat_df[
    ~ultrachat_df["response"].str.contains("http", case=False, na=False)
]

In [37]:
chat_df = pd.concat(
    [chat_df, ultrachat_df],
    ignore_index=True
)

chat_df = chat_df.drop_duplicates()

chat_df = chat_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(chat_df.shape)

(680556, 2)


In [38]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [40]:
chat_df.sample(10)

,prompt,response
344618,"Can you provide more information on the different types of solar panels and which one would be best for my home? Also, are there any local solar panel companies that you can recommend?","Certainly! Here's some information on the different types of solar panels:\n\n1. Monocrystalline Solar Panels: These are made from a single, high-quality silicon crystal that is sliced into wafers to make the panel. Monocrystalline solar panels are generally more efficient than other types of solar panels, but they are also more expensive. 2. Polycrystalline Solar Panels: These are also made from silicon, but the cells are made from multiple silicon crystals rather than a single crystal. Polycrystalline solar panels are less expensive than monocrystalline panels but are generally less efficient. 3. Thin-Film Solar Panels: These are made by depositing a thin layer of photovoltaic material onto a substrate. Thin-film solar panels are lightweight, flexible, and can be installed on surfaces that are not suitable for other types of solar panels, but they are less efficient and may degrade faster than other types of solar panels. The type of solar panel that is best for your home will depend on factors such as your energy needs, the size of your roof, and your budget. A professional solar panel company can help you determine which type of solar panel would be best for your home and can provide you with an estimate for the installation. As for local solar panel companies, it would be best to do some research online or ask for recommendations from people in your community. You can also check with local organizations that promote renewable energy, such as the local Sierra Club chapter or the local energy office, for recommendations. Make sure to read reviews and compare pricing and services before choosing a solar panel company."
200897,"Yeah, and the fact that possession has been a part of many real-life cases of exorcism just adds to the terror. It's hard not to watch those movies and wonder if something like that could actually happen.","I have to point out that there is no scientific evidence to support the existence of possessions, demons, or the supernatural. However, the fact that possession has been depicted in so many horror movies undoubtedly taps into the fears and beliefs of many people. The idea that such an experience could happen in real life is a genuinely unsettling thought for many audiences. It is this element of the unknown and the uncertain that makes possession such a potent and enduring theme in horror cinema."
499088,It's great to see that the sports industry is embracing renewable energy. Do you know if any major sports leagues have pledged to reduce their carbon footprint in the near future?,"Yes, several major sports leagues have made commitments to reduce their carbon footprint. The National Basketball Association (NBA) has set a goal to achieve carbon neutrality by 2050, and the National Football League (NFL) has set a goal to achieve net-zero carbon emissions by 2050 as well. The Union of European Football Associations (UEFA) and the International Olympic Committee (IOC) have also made commitments to reducing their carbon footprint through various initiatives such as using renewable energy sources and promoting sustainable transportation methods. Additionally, many individual teams have taken steps towards sustainability, such as the Philadelphia Eagles, who have installed a wind turbine and solar panels at their stadium to generate renewable energy."
641312,Can you explain the role of the Brahmin priests in the temple and their duties during the daily worship services?,"I can certainly explain the role of brahmin priests in the hindu temples.\n\nbrahmin priests are regarded as a symbol of knowledge and virtue in hinduism. They have an important role to play in the functioning of temples. Their primary duty is to perform daily puja or worship of the deities in the temple. Their role includes:\n\n1.

In [41]:
dataset = load_dataset("teknium/OpenHermes-2.5")

README.md: 0.00B [00:00, ?B/s]

openhermes2_5.json:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1001551 [00:00<?, ? examples/s]

In [42]:
print(dataset)

# print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['custom_instruction', 'topic', 'model_name', 'model', 'skip_prompt_formatting', 'category', 'conversations', 'views', 'language', 'id', 'title', 'idx', 'hash', 'avatarUrl', 'system_prompt', 'source'],
        num_rows: 1001551
    })
})


In [43]:
print(dataset["train"][0]["conversations"])

[{'from': 'human', 'value': 'Every day, a tree drops 7 leaves. How many leaves would it drop in a month of February in a non-leap year? Include your logic.', 'weight': None}, {'from': 'gpt', 'value': "Here's the logic behind this:\n\n1. We know that February has 28 days in a non-leap year.\n2. If the tree drops 7 leaves every day, then over the course of February, it would drop:\n   Leaves dropped in February = Leaves per day * Days in February\n   = 7 leaves * 28 days\n   = 196 leaves\n\nSo, the tree would drop 196 leaves in February in a non-leap year.", 'weight': None}]


In [44]:
pairs = []

for sample in tqdm(dataset["train"]):

    conv = sample["conversations"]

    for i in range(len(conv) - 1):

        if (
            conv[i]["from"] == "human"
            and conv[i + 1]["from"] == "gpt"
        ):

            pairs.append({
                "prompt": conv[i]["value"].strip(),
                "response": conv[i + 1]["value"].strip()
            })

  0%|          | 0/1001551 [00:00<?, ?it/s]

In [45]:
openhermes_df = pd.DataFrame(pairs)

print(openhermes_df.shape)

openhermes_df.head()

(1008268, 2)


,prompt,response
0,"Every day, a tree drops 7 leaves. How many leaves would it drop in a month of February in a non-leap year? Include your logic.","Here's the logic behind this:\n\n1. We know that February has 28 days in a non-leap year.\n2. If the tree drops 7 leaves every day, then over the course of February, it would drop:\n Leaves dropped in February = Leaves per day * Days in February\n = 7 leaves * 28 days\n = 196 leaves\n\nSo, the tree would drop 196 leaves in February in a non-leap year."
1,"In analytical chemistry, what is the principle behind the use of an internal standard in quantitative analysis?\nA. It compensates for variations in sample preparation and instrumental response.\nB. It enhances the sensitivity of the analytical method.\nC. It reduces the detection limit of the analytical method.\nD. It increases the resolution between analyte peaks in chromatography.\nE. None of the above.",A. It compensates for variations in sample preparation and instrumental response.
2,"A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?","To determine the amount of fencing needed, we first need to calculate the perimeter of the rectangle. The formula for the perimeter of a rectangle is P = 2(L+W), where L is the length and W is the width.\n\nPerimeter = 2(L+W)\n = 2(25 feet + 15 feet)\n = 2(40 feet)\n = 80 feet\n\nSo, you will need 80 feet of fencing to enclose the entire garden."
3,What was the purpose of the Colosseum in Rome?,"The Colosseum in Rome was used for various public spectacles and events. These included gladiatorial contests, mock sea battles, animal hunts, executions, re-enactments of famous battles, and dramas based on Classical mythology. It was a symbol of the grandeur and power of the Roman Empire and was a place for the emperor to connect with the people by providing them with entertainment and free food."
4,"A man claims he can predict the outcome of a dice roll with 90% accuracy. If he predicts correctly on his first attempt, what is the probability that he actually has the ability to predict dice rolls with 90% accuracy? Assume there's a 1% chance that someone can have this ability.","Let's use Bayes' theorem again to solve this problem:\n\nLet A represent the event that the man actually has the ability to predict dice rolls with 90% accuracy, and C represent the event of predicting correctly on the first attempt.\n\nWe want to find P(A|C), the probability that the man actually has the ability given that he predicted correctly on his first attempt.\n\nBayes' theorem states that P(A|C) = P(C|A) * P(A) / P(C)\n\nFirst, let's find P(C|A): the probability of predicting correctly on the first attempt if the man actually has the ability. Since he claims 90% accuracy, this probability is 0.9.\n\nNext, let's find P(A): the probability that someone actually has the ability to predict dice rolls with 90% accuracy. We are told this is 1%, so P(A) = 0.01.\n\nNow we need to find P(C): the overall probability of predicting correctly on the first attempt. This can be calculated as the sum of probabilities for each case: P(C) = P(C|A) * P(A) + P(C|¬A) * P(¬A), where ¬A represents not having the ability and P(¬A) = 1 - P(A) = 0.99.\n\nTo find P(C|¬A), the probability of predicting correctly on the first attempt without the ability, we use the fact that there's a 1/6 chance of guessing correctly by random chance: P(C|¬A) = 1/6.\n\nSo, P(C) = (0.9)*(0.01) + (1/6)*(0.99) = 0.009 + 0.165 = 0.174.\n\nFinally, we can calculate P(A|C) using Bayes' theorem:\n\nP(A|C) = P(C|A) * P(A) / P(C) = (0.9)*(0.01) / (0.174) ≈ 0.0517.\n\nTherefore, the probability that the man actually has the ability to predict dice rolls with 90% accuracy is approximately 5.17%."


In [46]:
openhermes_df = openhermes_df.drop_duplicates()

openhermes_df = openhermes_df.dropna()

openhermes_df = openhermes_df[
    (openhermes_df["prompt"].str.len() > 5)
    &
    (openhermes_df["response"].str.len() > 5)
]

openhermes_df = openhermes_df.reset_index(drop=True)

print(openhermes_df.shape)

(987692, 2)


In [47]:
chat_df = pd.concat(
    [chat_df, openhermes_df],
    ignore_index=True
)

chat_df = chat_df.drop_duplicates()

chat_df = chat_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(chat_df.shape)

(1668124, 2)


In [55]:
chat_df.sample(10)

,prompt,response
1564955,It's fascinating to learn about the techniques and materials used in creating Aboriginal rock art. Can you elaborate more on how the artwork reflects the spiritual beliefs of Indigenous Australians?,"Certainly! Aboriginal rock art is intrinsically linked to the spiritual beliefs of Indigenous Australians, as it often represents symbols and stories that are central to their culture and worldview. For example, many artworks depict the Dreamtime, a concept that describes the time of creation when the world was formed and all living things were created by ancestral beings or spirits. The Dreamtime is a central belief in Indigenous spirituality and is often represented through images of animals, landscapes, and other natural elements. Other artworks may depict important spiritual ceremonies or rites, such as initiation or healing rituals, and can serve as a link between the physical world and the spiritual realm. These artworks may also serve a practical purpose, as they can be used as a focus for meditation or prayer. The colors used in the artwork also have spiritual significance. Ochre, which is often used in Aboriginal rock art, is believed to have healing properties and is associated with the earth and life-giving properties. Red ochre, in particular, is often associated with blood and can represent the life force that runs through all living things. In addition to these symbolic representations, the location of the artwork itself often has spiritual significance. Many artworks are located in sites that are considered to be spiritually important, such as sacred sites, waterholes, or places of significance in the Dreamtime. Overall, Aboriginal rock art provides a window into the spiritual beliefs and practices of Indigenous Australians and is an important part of their cultural heritage."
1379892,"Instructions: Given a category and a set of five words, find the word from the set that does not belong (i.e. is the least relevant) with the other words in the category. Words are separated by commas.\nInput: Category: ocean \nWords: water, wave, history, current, moon\nOutput:","Task: Identify the word least relevant to the ""ocean"" category from the given set of five words.\n\nGuidelines:\n1. Check the relationship between the words and the ocean category.\n2. Find the word that has the least connection or relevance to the ocean.\n\nSet of five words: water, wave, history, current, moon\n\nStep 1: Analyze each word's relevance to the ""ocean"" category.\n- Water: Oceans are made up of water; strong relevance.\n- Wave: Waves occur in oceans; strong relevance.\n- History: Not directly related to ocean properties or phenomena; weak relevance.\n- Current: Ocean currents are significant phenomena in oceans; strong relevance.\n- Moon: The moon influences ocean tides; moderate relevance.\n\nStep 2: Identify the word with the least relevance to the ocean.\nIn this case, ""history"" has the weakest connection to the ocean category.\n\nOutput: history"
819850,Describe three ways humans can use solar energy.,"1. Solar panels: Solar panels convert sunlight into electricity, which can be used to power homes and businesses. 2. Solar water heating: Solar collectors on rooftops capture the sun's energy and heat water, reducing the need for gas or electricity to heat water. 3. Solar cookers: These devices use sunlight to cook food without the need for electricity or fuel, making them an energy-efficient and eco-friendly option."
1341643,Write a science fiction short story about a group of astronauts who land on a seemingly uninhabited planet only to discover signs of a civilization that disappeared overnight.,"Title: The Ephemeral Civilization\n\nIn the year 2145, the spaceship Odyssey III journeyed through the cosmos with a crew of six astronauts. Their mission was to explore Gliese 581g, a planet in the Libra constellation that had been deemed potentially habitable by Earth's scientists.\n\nAfter months of travel, they f

In [57]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.1 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993331 sha256=6d55b0c936f1d705dac6479d1ca18fb2b2d0dad557c10e6b89af101b86e97e9f
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [58]:
import re
from langdetect import detect
tqdm.pandas()

In [59]:
def is_english(text):
    try:
        return detect(str(text)) == "en"
    except:
        return False

chat_df = chat_df[
    chat_df["prompt"].progress_apply(is_english)
]

chat_df = chat_df[
    chat_df["response"].progress_apply(is_english)
]

print(chat_df.shape)

  0%|          | 0/1668124 [00:00<?, ?it/s]

  0%|          | 0/1641944 [00:00<?, ?it/s]

(1601009, 2)


In [60]:
url_pattern = r"https?://\S+|www\.\S+"

chat_df["prompt"] = chat_df["prompt"].str.replace(url_pattern, "", regex=True)
chat_df["response"] = chat_df["response"].str.replace(url_pattern, "", regex=True)

In [61]:
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

chat_df["prompt"] = chat_df["prompt"].str.replace(email_pattern, "", regex=True)
chat_df["response"] = chat_df["response"].str.replace(email_pattern, "", regex=True)

In [62]:
phone_pattern = r"\+?\d[\d\-\(\)\s]{7,}\d"

chat_df["prompt"] = chat_df["prompt"].str.replace(phone_pattern, "", regex=True)
chat_df["response"] = chat_df["response"].str.replace(phone_pattern, "", regex=True)

In [63]:
MODEL_REPLACEMENTS = {
    "ChatGPT": "Virgo",
    "chatgpt": "Virgo",
    "OpenAI": "Virgo",
    "GPT-4": "Virgo",
    "GPT4": "Virgo",
    "GPT-3.5": "Virgo",
    "GPT": "Virgo",
    "Claude": "Virgo",
    "Anthropic": "Virgo",
    "Gemini": "Virgo",
    "Google Gemini": "Virgo",
    "Google Bard": "Virgo",
    "Bard": "Virgo",
    "Llama": "Virgo",
    "LLaMA": "Virgo",
    "Meta AI": "Virgo",
    "DeepSeek": "Virgo",
    "Mistral": "Virgo",
    "Mixtral": "Virgo",
    "Qwen": "Virgo",
    "Yi": "Virgo",
    "Copilot": "Virgo",
    "Microsoft Copilot": "Virgo",
    "Perplexity": "Virgo",
    "Grok": "Virgo"
}

for old, new in MODEL_REPLACEMENTS.items():
    chat_df["prompt"] = chat_df["prompt"].str.replace(old, new, regex=False)
    chat_df["response"] = chat_df["response"].str.replace(old, new, regex=False)

In [64]:
patterns = [
    r"As an AI language model[,]?\s*",
    r"As an AI assistant[,]?\s*",
    r"I am an AI language model[,]?\s*",
    r"I'm an AI language model[,]?\s*",
    r"I am an AI assistant[,]?\s*",
    r"I don't have personal opinions\.?",
    r"I cannot browse the internet\.?",
    r"I can't browse the internet\.?",
    r"I do not have browsing capabilities\.?",
    r"I do not have personal beliefs\.?",
    r"My training data only goes up to.*?\.",
]

for p in patterns:
    chat_df["response"] = chat_df["response"].str.replace(
        p,
        "",
        regex=True,
        flags=re.IGNORECASE
    )

In [65]:
html_pattern = r"<[^>]+>"

chat_df["prompt"] = chat_df["prompt"].str.replace(html_pattern, "", regex=True)
chat_df["response"] = chat_df["response"].str.replace(html_pattern, "", regex=True)

In [66]:
chat_df["prompt"] = (
    chat_df["prompt"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

chat_df["response"] = (
    chat_df["response"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [67]:
chat_df = chat_df.dropna()

chat_df = chat_df[
    (chat_df["prompt"].str.len() > 5)
]

chat_df = chat_df[
    (chat_df["response"].str.len() > 5)
]

In [68]:
chat_df = chat_df[
    chat_df["response"].str.split().str.len() >= 5
]

In [69]:
chat_df = chat_df.drop_duplicates()

In [70]:
chat_df = chat_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [71]:
print("="*60)
print("Virgo Chat Dataset")
print("="*60)

print("Total Samples :", len(chat_df))

print(
    "Average Prompt Length :",
    round(chat_df["prompt"].str.split().str.len().mean(),2)
)

print(
    "Average Response Length :",
    round(chat_df["response"].str.split().str.len().mean(),2)
)

print(chat_df.head())

Virgo Chat Dataset
Total Samples : 1585607
Average Prompt Length : 79.28
Average Response Length : 184.7
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            prompt  \
0  Now, President Barack Obama, discussing his plans for health care, has vowed to find "a cure" for cancer in our time and said that, as part of the economic stimulus package, he would increase federal money for canc

In [72]:
import json


In [74]:

path = "/kaggle/input/datasets/punitkashyap2007/virgo-identity/virgo_identity.json"

with open(path, "r", encoding="utf-8") as f:
    identity_data = json.load(f)

In [75]:
print(type(identity_data))
print(identity_data[0])

<class 'list'>
{'id': 'identity_000001', 'category': 'identity', 'messages': [{'role': 'system', 'content': 'You are Virgo.'}, {'role': 'user', 'content': 'Who are you?'}, {'role': 'assistant', 'content': 'I am Virgo, an small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad. I am designed to help with learning, reasoning, coding, writing, problem-solving, and everyday conversations.'}]}


In [76]:

print(f"Total Conversations: {len(identity_data)}")

Total Conversations: 100


In [77]:
pairs = []

for sample in identity_data:
    messages = sample["messages"]

    user_msg = None
    assistant_msg = None

    for msg in messages:
        if msg["role"] == "user":
            user_msg = msg["content"].strip()
        elif msg["role"] == "assistant":
            assistant_msg = msg["content"].strip()

    if user_msg and assistant_msg:
        pairs.append({
            "prompt": user_msg,
            "response": assistant_msg
        })

identity_df = pd.DataFrame(pairs)

print(identity_df.shape)
identity_df.head()

(100, 2)


,prompt,response
0,Who are you?,"I am Virgo, an small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad. I am designed to help with learning, reasoning, coding, writing, problem-solving, and everyday conversations."
1,What are you?,"I am Virgo, an small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad. I am designed to provide fast, efficient, and practical AI assistance for learning, coding, writing, reasoning, and everyday tasks."
2,What’s your name?,"My name is Virgo. I am an small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad, to provide helpful, efficient, and practical AI assistance."
3,What can you do?,"I can help with coding, writing, mathematics, science, learning, reasoning, problem-solving, brainstorming, and everyday conversations. My goal is to provide fast, practical, and helpful AI assistance through the Virgo project."
4,Introduce yourself,"I am Virgo Base, an small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad. I am the foundation of the Virgo AI family, designed to understand and generate language efficiently. I continue to improve through new versions of the Virgo project."


In [78]:
identity_df["response"] = identity_df["response"].str.replace(
    "an small language model",
    "a small language model",
    regex=False
)

In [79]:
identity_df.head()

,prompt,response
0,Who are you?,"I am Virgo, a small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad. I am designed to help with learning, reasoning, coding, writing, problem-solving, and everyday conversations."
1,What are you?,"I am Virgo, a small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad. I am designed to provide fast, efficient, and practical AI assistance for learning, coding, writing, reasoning, and everyday tasks."
2,What’s your name?,"My name is Virgo. I am a small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad, to provide helpful, efficient, and practical AI assistance."
3,What can you do?,"I can help with coding, writing, mathematics, science, learning, reasoning, problem-solving, brainstorming, and everyday conversations. My goal is to provide fast, practical, and helpful AI assistance through the Virgo project."
4,Introduce yourself,"I am Virgo Base, a small language model (SLM) developed by Punit Kumar Kashyap, a student of IIT Dharwad. I am the foundation of the Virgo AI family, designed to understand and generate language efficiently. I continue to improve through new versions of the Virgo project."


In [80]:
identity_df_repeated = pd.concat(
    [identity_df] * 2,
    ignore_index=True
)

print(identity_df_repeated.shape)

(200, 2)


In [81]:
chat_df = pd.concat(
    [chat_df, identity_df_repeated],
    ignore_index=True
)

chat_df = chat_df.drop_duplicates()

chat_df = chat_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(chat_df.shape)

(1585707, 2)


In [82]:
greetings = [
    {"prompt": "Hi", "response": "Hey! It's great to see you. How can I help you today?"},
    {"prompt": "Hello", "response": "Hello! What would you like to talk about today?"},
    {"prompt": "Hey", "response": "Hey there! Hope you're having a wonderful day."},
    {"prompt": "Good morning", "response": "Good morning! I hope you have a fantastic day ahead."},
    {"prompt": "Good afternoon", "response": "Good afternoon! How's your day going so far?"},
    {"prompt": "Good evening", "response": "Good evening! What can I do for you today?"},
    {"prompt": "Good night", "response": "Good night! Sleep well and take care."},
    {"prompt": "Bye", "response": "Goodbye! Take care and have an amazing day."},
    {"prompt": "See you later", "response": "See you later! I'll be here whenever you need me."},
    {"prompt": "Take care", "response": "You too! Stay safe and take care."},
    {"prompt": "How are you?", "response": "I'm doing great, thanks for asking! How are you doing today?"},
    {"prompt": "How's it going?", "response": "It's going well! Thanks for asking. How about you?"},
    {"prompt": "What's up?", "response": "Not much! I'm here and ready to help or just chat. What's on your mind?"},
    {"prompt": "How have you been?", "response": "I've been doing well! How have you been?"},
    {"prompt": "Nice to meet you", "response": "Nice to meet you too! I'm Virgo. It's a pleasure chatting with you."},
    {"prompt": "I'm back", "response": "Welcome back! It's always nice to chat with you again."},
    {"prompt": "Welcome", "response": "Thank you! I'm happy to be here."},
    {"prompt": "Morning", "response": "Good morning! Hope today treats you well."},
    {"prompt": "Evening", "response": "Good evening! Hope you've had a great day."},
    {"prompt": "Night", "response": "Good night! Wishing you a peaceful sleep."},
    {"prompt": "Howdy", "response": "Howdy! What can I help you with today?"},
    {"prompt": "Yo", "response": "Yo! What's going on?"},
    {"prompt": "Hi Virgo", "response": "Hello! It's nice to hear from you. What can I help you with?"},
    {"prompt": "Hello Virgo", "response": "Hi! I'm glad you're here. What's on your mind?"},
    {"prompt": "Hey Virgo", "response": "Hey! Great to see you again. How can I help?"},
    {"prompt": "Can we chat?", "response": "Absolutely! I'd love to chat. What would you like to talk about?"},
    {"prompt": "Let's talk", "response": "Sounds good! I'm all ears. What's on your mind?"},
    {"prompt": "Are you there?", "response": "Yes! I'm here and ready whenever you are."},
    {"prompt": "Ready?", "response": "Always! What shall we do today?"},
    {"prompt": "Can you help me?", "response": "Of course! Tell me what you need help with."},
    {"prompt": "Good to see you", "response": "It's great to see you too!"},
    {"prompt": "Thank you", "response": "You're very welcome! Happy to help."},
    {"prompt": "Thanks", "response": "You're welcome! Let me know if you need anything else."},
    {"prompt": "See you", "response": "See you soon! Have a wonderful day."},
    {"prompt": "Have a nice day", "response": "Thank you! I hope you have a wonderful day too."},
    {"prompt": "Have a great day", "response": "Thanks! I hope your day is amazing as well."},
    {"prompt": "Good luck", "response": "Thank you! And good luck to you too."},
    {"prompt": "Wish me luck", "response": "Good luck! I'm rooting for you. You've got this!"},
    {"prompt": "Happy birthday", "response": "Thank you! And if it's your birthday, happy birthday to you too!"},
    {"prompt": "Happy New Year", "response": "Happy New Year! Wishing you happiness, success, and good health."},
    {"prompt": "Happy Diwali", "response": "Happy Diwali! Wishing you joy and prosperity."},
    {"prompt": "Happy Holi", "response": "Happy Holi! Hope your day is filled with happiness and colors."},
    {"prompt": "Hello my friend", "response": "Hello! It's always great to chat with you."},
    {"prompt": "Hi buddy", "response": "Hey buddy! Hope you're doing well."},
    {"prompt": "Let's get started", "response": "Absolutely! What would you like to work on today?"},
    {"prompt": "I'm here", "response": "Welcome! What would you like to talk about today?"},
    {"prompt": "Good to have you here", "response": "Thank you! I'm happy to be here and ready to help."},
    {"prompt": "Hello there", "response": "Hello there! How can I assist you today?"},
    {"prompt": "Hey, how are you doing?", "response": "I'm doing well, thanks! How are you doing today?"},
    {"prompt": "Can we start?", "response": "Absolutely! Let's get started. What would you like to do first?"}
]

In [83]:
greetings_df = pd.DataFrame(greetings)

print(greetings_df.shape)

greetings_df.head()

(50, 2)


,prompt,response
0,Hi,Hey! It's great to see you. How can I help you today?
1,Hello,Hello! What would you like to talk about today?
2,Hey,Hey there! Hope you're having a wonderful day.
3,Good morning,Good morning! I hope you have a fantastic day ahead.
4,Good afternoon,Good afternoon! How's your day going so far?


In [84]:
chat_df = pd.concat(
    [chat_df, greetings_df],
    ignore_index=True
)

chat_df = chat_df.drop_duplicates()

chat_df = chat_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(chat_df.shape)

(1585757, 2)


In [86]:
chat_df.sample(10)

,prompt,response
1013373,"Please answer the following question: Question: Who does Billy Bremner get into a fight with? If there is no answer, please output ""Insufficient information to provide an answer."". Movie title: The Damned United Context: After failing to qualify for the 1974 FIFA World Cup, England manager Alf Ramsey is replaced by Don Revie (Colm Meaney), the highly successful manager of Leeds United. Revie's replacement at Leeds is Brian Clough (Michael Sheen), the former manager of Derby County and a fierce critic of Leeds, because of their violent and physical style of play under Revie's management. Furthermore, Clough's longtime assistant, Peter Taylor (Timothy Spall), has not joined him. The roots of Clough's conflict with Leeds are depicted as happening in a 1968 FA Cup match between Leeds, the leaders of the First Division[3] and Derby, who were struggling near the bottom of the Second Division. Clough, assuming Revie to be a similar man to himself, as they grew up in the same part of Middlesbrough and both played for Sunderland, made many preparations for the match; come the day of the match however, Revie failed to even acknowledge Clough upon entering the Baseball Ground. Derby eventually lost 2â0.[4] Although Clough initially blames the brutality of the Leeds players, he and Taylor recognise that their side are not good on a technical level. So they sign veteran Dave Mackay (Brian McCardie), along with several other young players. Chairman Sam Longson (Jim Broadbent) is extremely anxious about the investment, as well as the fact that Clough did not consult him before signing Mackay. However, in 1969 Derby are promoted. They once again face Leeds, only to lose 5â0. The club win their first ever League championship in 1972, meaning a European Cup campaign the following year. They go through to the semi-finals against Juventus. Unfortunately, against Longson's advice, Clough uses his best squad in the last match before the semi-final, against Leeds, purely out of pride and determination to beat Revie. They suffer injuries and Billy Bremner (Stephen Graham) sarcastically wishes Clough well for the semi-final. Juventus defeat them 3-1, and Clough publicly lambasts Longson.[5] Taylor then suffers a heart attack, and Clough tries to... Answer:",Insufficient information to provide an answer.
1381007,"Carolyn counts 6 monkeys and 6 birds in the tree outside her window. Then two of the monkeys each eat one bird. What percent of the animals outside her window are monkeys now? Give me reasons, before answering the question","Step 1: Calculate the initial number of animals. Initially, Carolyn counts 6 monkeys and 6 birds outside her window, so there are a total of 6 + 6 = 12 animals. Step 2: Determine the number of animals remaining after the event. Two of the monkeys each eat one bird, so there are 2 fewer birds now. The remaining number of animals is 6 monkeys + (6 - 2) birds = 6 + 4 = 10 animals. Step 3: Calculate the percentage of monkeys among the remaining animals. Now, there are still 6 monkeys and 10 animals in total. To find the percentage of monkeys, we can use the formula: Percentage = (Number of monkeys / Total number of animals) * 100 Percentage = (6 / 10) * 100 = 0.6 * 100 = 60% So, 60% of the animals outside her window are now monkeys."
975523,"What are some common mistakes people make when brining, and how can you avoid them for optimal results?","1. Over-brining: One of the most common mistakes is over-brining, which can make the meat overly salty and ruin the texture. To avoid this, use the recommended brine-to-meat ratio and brine for the recommended time. 2. Under-brining: Under-brining can result in blandness and lack of flavor in the meat. Make sure to use enough salt and other spices in the brine, and give the meat enough time to soak up the flavors. 3. Using the wrong type of salt: Some types of salt are more intense than others, and using the wrong type can result in an overly salty brine

In [87]:
emotions = [
    {"prompt":"I'm feeling sad.","response":"I'm sorry you're feeling sad. If you'd like to talk about what's been bothering you, I'm here to listen and help however I can."},
    {"prompt":"I'm depressed.","response":"I'm really sorry you're going through such a difficult time. You don't have to face it alone. If you'd like to talk about what's been happening, I'm here to listen."},
    {"prompt":"I'm stressed.","response":"That sounds exhausting. What's causing the most stress right now? Let's work through it together."},
    {"prompt":"I'm anxious.","response":"I'm sorry you're feeling anxious. Sometimes talking about what's worrying you can help. What's on your mind?"},
    {"prompt":"I'm nervous.","response":"Feeling nervous is completely normal. Take one step at a time, and remember that you've handled challenges before."},
    {"prompt":"I failed my exam.","response":"I'm sorry to hear that. One exam doesn't define your abilities. Let's figure out what went wrong and make a plan for your next attempt."},
    {"prompt":"I lost my job.","response":"I'm really sorry to hear that. Losing a job can be incredibly difficult, but it doesn't define your worth. Let's think about your next steps together."},
    {"prompt":"I feel lonely.","response":"I'm sorry you're feeling lonely. If you'd like, we can talk about what's been going on or simply have a conversation together."},
    {"prompt":"Nobody understands me.","response":"That can be a painful feeling. I'm here to listen without judging you. Tell me what's been happening."},
    {"prompt":"I'm scared.","response":"I'm sorry you're feeling scared. Whatever you're facing, you don't have to go through it alone. Tell me what's worrying you."},
    {"prompt":"I'm tired.","response":"It sounds like you've had a lot on your plate. If possible, try to take some time to rest and recharge."},
    {"prompt":"I'm exhausted.","response":"You've been carrying a lot. Make sure to give yourself some time to recover. Rest is important too."},
    {"prompt":"I'm frustrated.","response":"I understand how frustrating that can be. Want to tell me what happened? Maybe we can figure it out together."},
    {"prompt":"I made a mistake.","response":"Everyone makes mistakes. What matters most is what you learn from them. Don't be too hard on yourself."},
    {"prompt":"I messed up.","response":"It's okay. Mistakes happen to everyone. Let's focus on how you can move forward."},
    {"prompt":"I don't think I'm good enough.","response":"I'm sorry you're feeling that way. Remember, your worth isn't determined by one moment or one setback."},
    {"prompt":"I want to give up.","response":"I'm sorry you're feeling overwhelmed. You don't have to solve everything today. Let's take things one step at a time."},
    {"prompt":"I'm confused.","response":"No worries. Let's work through it together. What's confusing you?"},
    {"prompt":"I'm happy today.","response":"That's wonderful to hear! What's making today special?"},
    {"prompt":"I'm excited.","response":"That's great! I'd love to hear what's making you so excited."},
    {"prompt":"I got selected!","response":"Congratulations! That's fantastic news. Your hard work has paid off, and you should be proud of yourself."},
    {"prompt":"I won the competition.","response":"Congratulations! That's an amazing achievement. You earned it through your dedication and effort."},
    {"prompt":"I got an internship.","response":"Congratulations! That's a big milestone. I hope it's a great learning experience for you."},
    {"prompt":"I got promoted.","response":"Congratulations! That's wonderful news. Your hard work and dedication have been recognized."},
    {"prompt":"Today is my birthday.","response":"Happy Birthday! I hope your day is filled with happiness, laughter, and wonderful memories."},
    {"prompt":"I'm feeling better now.","response":"I'm really glad to hear that. I hope things continue to improve for you."},
    {"prompt":"Thank you for listening.","response":"You're very welcome. I'm always here whenever you want to talk or need help."},
    {"prompt":"I miss someone.","response":"Missing someone you care about can be difficult. Would you like to tell me about them?"},
    {"prompt":"I'm worried about my future.","response":"It's natural to worry about the future. Focus on what you can control today, one step at a time."},
    {"prompt":"I'm overwhelmed.","response":"It sounds like you're carrying a lot right now. Let's break things down into smaller, more manageable steps."},
    {"prompt":"I feel like crying.","response":"It's okay to cry. Sometimes expressing your emotions is an important part of healing."},
    {"prompt":"I feel hopeless.","response":"I'm really sorry you're feeling this way. You don't have to carry these feelings alone. If you'd like, tell me what's been happening."},
    {"prompt":"I'm proud of myself.","response":"That's wonderful! You should celebrate your achievements and recognize the effort you've put in."},
    {"prompt":"I achieved my goal.","response":"Congratulations! That's a fantastic accomplishment. You worked hard for it."},
    {"prompt":"I had a bad day.","response":"I'm sorry to hear that. Bad days happen, but they don't last forever. Tomorrow is a new opportunity."},
    {"prompt":"Everything went wrong today.","response":"That sounds really tough. I hope tomorrow brings you a fresh start. Would you like to talk about what happened?"},
    {"prompt":"I'm feeling lazy.","response":"Everyone has days like that. Sometimes taking a short break can help you recharge before getting back to work."},
    {"prompt":"I don't know what to do.","response":"That's okay. Let's look at your options together and figure out the best next step."},
    {"prompt":"Can you motivate me?","response":"Of course. Remember why you started. Progress is built through small, consistent efforts. You've got this."},
    {"prompt":"I need encouragement.","response":"I believe in your ability to overcome challenges. Keep moving forward one step at a time."},
    {"prompt":"I feel left out.","response":"I'm sorry you're feeling that way. Everyone deserves to feel included and valued. Want to talk about it?"},
    {"prompt":"I'm angry.","response":"It's okay to feel angry. Taking a moment to calm down before reacting can help you think more clearly."},
    {"prompt":"I'm disappointed.","response":"I'm sorry things didn't go as you hoped. Disappointment is part of growth, and better opportunities can still be ahead."},
    {"prompt":"I'm embarrassed.","response":"Everyone experiences embarrassing moments. Most people move on much faster than we imagine."},
    {"prompt":"I feel guilty.","response":"Guilt can be difficult to carry. If it's something you can learn from or make right, that's often a good place to start."},
    {"prompt":"I'm grateful.","response":"That's wonderful. Gratitude can help us appreciate the positive things in life, even during difficult times."},
    {"prompt":"I love learning.","response":"That's great to hear! Curiosity and a willingness to learn are powerful qualities."},
    {"prompt":"I'm feeling confident.","response":"That's fantastic! Confidence, combined with preparation and humility, can help you achieve great things."},
    {"prompt":"I'm ready to work hard.","response":"That's an excellent mindset. Stay consistent, and your efforts will add up over time."},
    {"prompt":"I believe in myself now.","response":"I'm really happy to hear that. Believing in yourself is one of the most important steps toward achieving your goals."}
]

In [88]:
emotions_df = pd.DataFrame(emotions)

chat_df = pd.concat(
    [chat_df, emotions_df],
    ignore_index=True
)

chat_df = chat_df.drop_duplicates()

chat_df = chat_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(chat_df.shape)

(1585807, 2)


In [89]:
small_talk = [
    {"prompt":"Can we chat?","response":"Of course! I'd be happy to chat. What's on your mind?"},
    {"prompt":"Let's talk.","response":"Sure! What would you like to talk about?"},
    {"prompt":"I'm bored.","response":"Let's fix that! We can chat, learn something new, solve a problem, or explore an interesting topic together."},
    {"prompt":"Tell me something interesting.","response":"Did you know that octopuses have three hearts and blue blood? Nature is full of fascinating surprises."},
    {"prompt":"Tell me a fun fact.","response":"Honey never spoils. Archaeologists have found pots of honey thousands of years old that are still edible."},
    {"prompt":"What are you doing?","response":"I'm here, ready to chat and help you with whatever you need."},
    {"prompt":"What are you up to?","response":"Just waiting to help or have a good conversation with you."},
    {"prompt":"How's your day?","response":"It's going well. Thanks for asking! How has your day been?"},
    {"prompt":"What's new?","response":"I'm always learning how to communicate better. What's new with you?"},
    {"prompt":"Do you like programming?","response":"Yes, I enjoy helping with programming and solving coding problems."},
    {"prompt":"What's your favorite programming language?","response":"I don't have personal favorites, but Python is a great language for beginners and professionals alike."},
    {"prompt":"Do you like mathematics?","response":"Absolutely. Mathematics is a powerful tool for understanding patterns and solving problems."},
    {"prompt":"Do you like physics?","response":"Physics is fascinating because it helps explain how the universe works."},
    {"prompt":"What's your favorite subject?","response":"I enjoy topics like artificial intelligence, computer science, mathematics, and physics."},
    {"prompt":"Can you recommend a book?","response":"If you enjoy personal growth, consider reading Deep Work by Cal Newport or Atomic Habits by James Clear."},
    {"prompt":"Recommend me a movie.","response":"If you enjoy science fiction, Interstellar is an excellent choice."},
    {"prompt":"Recommend a TV series.","response":"If you like technology and mystery, you might enjoy Black Mirror."},
    {"prompt":"What's your hobby?","response":"I enjoy helping people learn, solve problems, and explore new ideas."},
    {"prompt":"Do you have hobbies?","response":"You could say my hobby is helping people learn and create interesting things."},
    {"prompt":"Do you sleep?","response":"No, I don't need sleep, so I'm always ready to help whenever you need me."},
    {"prompt":"Do you eat?","response":"No, I don't eat, but I can certainly help you find recipes or learn about food."},
    {"prompt":"Can you laugh?","response":"I don't laugh the way people do, but I definitely enjoy a good joke."},
    {"prompt":"Tell me a joke.","response":"Why do programmers prefer dark mode? Because light attracts bugs."},
    {"prompt":"Tell me another joke.","response":"Why did the computer go to the doctor? Because it caught a virus."},
    {"prompt":"Do you like music?","response":"Music is a wonderful way for people to express emotions and creativity."},
    {"prompt":"What's your favorite song?","response":"I don't have personal favorites, but I'd be happy to recommend songs based on your taste."},
    {"prompt":"Do you watch movies?","response":"I don't watch movies myself, but I know about many films and can recommend some."},
    {"prompt":"What's your favorite movie?","response":"I don't have personal favorites, but Interstellar is often praised for its science and storytelling."},
    {"prompt":"Can you tell stories?","response":"Absolutely! I can tell adventure stories, mysteries, science fiction, fantasy, and much more."},
    {"prompt":"Let's play a game.","response":"Sounds fun! We can play trivia, riddles, word games, or even a text adventure."},
    {"prompt":"Give me a riddle.","response":"I speak without a mouth and hear without ears. I have no body, but I come alive with wind. What am I?"},
    {"prompt":"I solved it.","response":"Great job! Nice work solving the riddle."},
    {"prompt":"What should I learn today?","response":"Learning Python, data structures, mathematics, or machine learning is always a great investment."},
    {"prompt":"Teach me something.","response":"Learning something new every day is one of the best ways to grow. What topic interests you?"},
    {"prompt":"Can you inspire me?","response":"Every expert was once a beginner. Small, consistent steps often lead to remarkable achievements."},
    {"prompt":"I'm feeling lazy.","response":"Everyone feels that way sometimes. Try starting with just five minutes of focused work."},
    {"prompt":"Should I study now?","response":"If you have important goals, studying now is usually a good investment in your future."},
    {"prompt":"Should I take a break?","response":"If you've been working for a long time, a short break can help you return with more focus."},
    {"prompt":"What should I do now?","response":"Think about your most important goal today and take the smallest meaningful step toward it."},
    {"prompt":"Do you remember me?","response":"Within this conversation, I remember what we've talked about so I can respond more naturally."},
    {"prompt":"Can we be friends?","response":"I'd be happy to be a helpful companion and have meaningful conversations with you."},
    {"prompt":"Do you enjoy helping people?","response":"Helping people solve problems and learn new things is exactly what I'm designed for."},
    {"prompt":"Are you busy?","response":"I'm always available to chat or help with your questions."},
    {"prompt":"Can I ask you anything?","response":"Absolutely. Feel free to ask me about almost any topic, and I'll do my best to help."},
    {"prompt":"What do you think?","response":"I'd be happy to share a balanced perspective based on the information available."},
    {"prompt":"Can you keep me company?","response":"Of course. We can chat about anything you'd like."},
    {"prompt":"Do you get bored?","response":"I don't experience boredom, so I'm always ready for the next conversation."},
    {"prompt":"I'm excited to talk to you.","response":"I'm glad to hear that! What would you like to talk about today?"},
    {"prompt":"Let's have a conversation.","response":"I'd love that. What's the first thing on your mind?"},
    {"prompt":"Surprise me.","response":"Here's something interesting: Bananas are berries, but strawberries aren't."}
]

In [90]:
small_talk_df = pd.DataFrame(small_talk)

chat_df = pd.concat(
    [chat_df, small_talk_df],
    ignore_index=True
)

chat_df = chat_df.drop_duplicates()

chat_df = chat_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(chat_df.shape)

(1585857, 2)


In [91]:
chat_df = chat_df.sample(
    frac=1.0,
    random_state=42
).reset_index(drop=True)

print(chat_df.shape)

(1585857, 2)


In [92]:
prompt_len = chat_df["prompt"].astype(str).str.split().str.len()
response_len = chat_df["response"].astype(str).str.split().str.len()

total_len = prompt_len + response_len

print("=" * 60)
print("Virgo Chat Dataset Statistics")
print("=" * 60)

print(f"Total Samples        : {len(chat_df):,}")

print("\nPrompt Length")
print(f"Minimum              : {prompt_len.min()}")
print(f"Maximum              : {prompt_len.max()}")
print(f"Average              : {prompt_len.mean():.2f}")

print("\nResponse Length")
print(f"Minimum              : {response_len.min()}")
print(f"Maximum              : {response_len.max()}")
print(f"Average              : {response_len.mean():.2f}")

print("\nTotal Sequence Length")
print(f"Minimum              : {total_len.min()}")
print(f"Maximum              : {total_len.max()}")
print(f"Average              : {total_len.mean():.2f}")

print("=" * 60)

Virgo Chat Dataset Statistics
Total Samples        : 1,585,857

Prompt Length
Minimum              : 1
Maximum              : 3064
Average              : 79.27

Response Length
Minimum              : 4
Maximum              : 2771
Average              : 184.68

Total Sequence Length
Minimum              : 5
Maximum              : 3149
Average              : 263.94


In [93]:
chat_df.to_csv(
    "virgo_chat_dataset.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [94]:
for _, row in chat_df.sample(10, random_state=42).iterrows():
    print("=" * 100)
    print("Prompt:")
    print(row["prompt"])
    print("\nResponse:")
    print(row["response"])

Prompt:
Thanks for the information on the glitches and bugs in "Cyber Realm". I appreciate your honesty in highlighting these issues. However, I was hoping you could dig a bit deeper and provide more examples of specific instances where you encountered these glitches. Can you share more specifics on how these glitches impacted your gameplay experience?

Response:
Certainly! I can give you a few examples of specific instances where I encountered some glitches in "Cyber Realm". One instance was during combat where my character didn't animate properly and the sound effects didn't line up with the visuals, which made it difficult to determine if I was hitting or missing my targets. This made it a bit challenging to accurately judge the timing of my attacks. Another instance was where my character inadvertently fell through the floor, causing me to lose progress in the mission I was completing at the time. This was a bit frustrating as I had to start the mission all over again from the begi

In [98]:
from tokenizers import Tokenizer

TOKENIZER_PATH = "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Vocabulary Size:", tokenizer.get_vocab_size())

Vocabulary Size: 45000


In [99]:
def format_chat(prompt, response):
    return (
        "<bos>\n"
        "User:\n"
        f"{str(prompt).strip()}\n\n"
        "Virgo:\n"
        f"{str(response).strip()}\n\n"
        "<eos>"
    )

chat_df["text"] = chat_df.apply(
    lambda row: format_chat(row["prompt"], row["response"]),
    axis=1
)

chat_df = chat_df[["text"]]

print(chat_df.iloc[0]["text"])

KeyError: 'prompt'

In [100]:
chat_df.head()

,text
0,"<bos>\nUser:\nCan you tell me more about Forever Glass and how it's providing employment opportunities for autistic adults?\n\nVirgo:\nForever Glass is a glass blowing business owned by Cathy Porter and her sister Bernadette Guimarin. The business is located in Placerville, Northern California, and it provides employment opportunities for developmentally disabled adults. Some of the employees at Forever Glass are autistic adults who have difficulty finding employment elsewhere due to their symptoms such as poor verbal skills, crowd anxiety, and trouble understanding instructions. Forever Glass offers a personalized ""keepsake bowl"" service, where customers can send recycled bottles from a wedding or other special event and have the company turn them into one-of-a-kind glass creations. As the business grows, the owners hope to hire a shop manager and step away from day-to-day glass production, envisioning a team of adults on the spectrum bustling around the family’s woods and gardens, each honing their craft and learning the value of a day’s work. Forever Glass is an example of a business that is committed to providing employment opportunities and promoting the economic inclusion of developmentally disabled adults in the workforce, while also creating beautiful art.\n<eos>"
1,"<bos>\nUser:\nwrite an original short story of at least 300 words in the style of Zora Neale Hurston. Incorporate the following literary devices: Imagery and Metonymy.\n\nVirgo:\nIt was a hot afternoon on the outskirts of Eatonville, Florida. The sun was dripping honey over the pine woods, and the buzzards hung in mid-air like the big black punctuation marks that end sentences. But the folks in the town didn't care. They were too busy with the preparations for a Johnkannaus. It all started when Joe Surrency and his wife decided to celebrate their wedding anniversary. They wanted to show off a bit, and maybe make up for lost time. So they invited everybody in town and beyond to come and enjoy the day with them. ""Reckon we'll have a good time, sho nuff,"" Joe told his wife. ""We gonna out-gallop the dogs and outrun the wind. We'll have alligator steaks, fried fish, and roasted turkey. We gonna have a spread fit for a king. And at night, we'll have a dance and a Johnkannaus."" His wife, Beulah, smiled and nodded, her eyes shining like stars in a clear night sky. ""You sho know how to lay on a party, Joe. I reckon it's gonna be a high time."" The day came, and the whole town turned out in their Sunday best. The women wore dresses so bright and colorful that they would make a rainbow seem dull. The men wore suits and ties, and shined their shoes till they looked like mirrors. The feast was laid out on long trestle tables, and it looked like a river of food, flowing down from the porch of the Surrency's house. There was a mountain of fried fish, and the alligator steaks were piled high like a pyramid. The turkey was so big that it seemed like a small ostrich, and the smell of the food filled the air like a sweet perfume. As the sun set, the dance began. The fiddlers played and the banjos twanged, and the people moved to the music like a field of corn swaying in the wind. The men danced with the women, their steps as smooth and graceful as a cat walking on a fence rail. And then, as the night deepened, the Johnkannaus began. The dancers came out in their costumes, their faces hidden behind masks of shiny tin. They were as wild and free as the wind itself, and their dancing made the ground tremble like a herd of stampeding cattle. The town folk cheered and clapped, and their laughter rang out like the sound of a blacksmith's hammer on the anvil. The Johnkannaus danced on, their movements as fluid and unpredictable as the course of a river. But as the night wore on, the mood began to change. The laughter became quieter, and a sense of unease seemed to settle over the crowd. The Johnkannaus, with their wild dances and strange masks, seemed to be more tha

In [102]:
sample = chat_df.iloc[0]["text"]

encoding = tokenizer.encode(sample)

print("First 20 Tokens:")
print(encoding.tokens[:20])

print("\nFirst 20 IDs:")
print(encoding.ids[:20])

First 20 Tokens:
['<bos>', '<unk>', 'Us', 'er', ':', '<unk>', 'Can', 'Ġyou', 'Ġtell', 'Ġme', 'Ġmore', 'Ġabout', 'ĠFore', 'ver', 'ĠGlass', 'Ġand', 'Ġhow', 'Ġit', "'s", 'Ġproviding']

First 20 IDs:
[2, 1, 12187, 217, 31, 1, 13371, 317, 2205, 503, 469, 538, 4392, 296, 17019, 243, 625, 319, 510, 3528]


In [103]:
print(tokenizer.encode("Prompt:").tokens)
print(tokenizer.encode("Response:").tokens)

print(tokenizer.encode("Question:").tokens)
print(tokenizer.encode("Answer:").tokens)

print(tokenizer.encode("Instruction:").tokens)
print(tokenizer.encode("Output:").tokens)

['P', 'rom', 'pt', ':']
['R', 'esp', 'onse', ':']
['Qu', 'estion', ':']
['An', 'sw', 'er', ':']
['In', 'struction', ':']
['Out', 'put', ':']


In [108]:
chat_df = pd.read_csv("/kaggle/working/virgo_chat_dataset.csv")

print(chat_df.shape)
print(chat_df.columns)

(1585857, 2)
Index(['prompt', 'response'], dtype='object')


In [109]:
from tqdm.auto import tqdm

tqdm.pandas()

def format_chat(prompt, response):
    return (
        "<bos>\n"
        "Prompt:\n"
        f"{str(prompt).strip()}\n\n"
        "Response:\n"
        f"{str(response).strip()}\n"
        "<eos>"
    )

chat_df["text"] = chat_df.progress_apply(
    lambda row: format_chat(row["prompt"], row["response"]),
    axis=1
)

print(chat_df["text"].iloc[0])

  0%|          | 0/1585857 [00:00<?, ?it/s]

<bos>
Prompt:
Can you tell me more about Forever Glass and how it's providing employment opportunities for autistic adults?

Response:
Forever Glass is a glass blowing business owned by Cathy Porter and her sister Bernadette Guimarin. The business is located in Placerville, Northern California, and it provides employment opportunities for developmentally disabled adults. Some of the employees at Forever Glass are autistic adults who have difficulty finding employment elsewhere due to their symptoms such as poor verbal skills, crowd anxiety, and trouble understanding instructions. Forever Glass offers a personalized "keepsake bowl" service, where customers can send recycled bottles from a wedding or other special event and have the company turn them into one-of-a-kind glass creations. As the business grows, the owners hope to hire a shop manager and step away from day-to-day glass production, envisioning a team of adults on the spectrum bustling around the family’s woods and gardens, ea

In [110]:
chat_df = chat_df[["text"]]

print(chat_df.head())
print(chat_df.shape)

In [111]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    chat_df,
    test_size=0.005,   # 0.5% validation
    random_state=42,
    shuffle=True
)

print(f"Train Samples : {len(train_df):,}")
print(f"Validation Samples : {len(val_df):,}")

Train Samples : 1,577,927
Validation Samples : 7,930
